# **Cyber Bullying Detection**

By Using GapHate Corpus we imported data of catigorized tweets to train the data:

-vo : violent language or context

-hd : hate speech

-cv : esplicit calls to violence

In [1]:
!pip install scikit-learn pandas

In [2]:
import pandas as pd

In [3]:
train = pd.read_csv('ghc_train.tsv', sep='\t')
test = pd.read_csv('ghc_test.tsv', sep='\t')
test.sample(5)

,text,hd,cv,vo
1764,"the ""Ninth Circle Satanic Child Sacrifice Cult...",0,0,0
465,How to Prevent Gallstones When You Have Crohn'...,0,0,0
3018,Question is why are IRS agents combat trained ?,0,0,0
1562,"I'm not insulting Creationism, I'm showing how...",0,0,0
4905,"My family escaped socialism, now my fellow Dem...",0,0,0


In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer #
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier


In [5]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import SGDClassifier

In [6]:
vectorizer = TfidfVectorizer()
x_train_vec = vectorizer.fit_transform(train['text'])

y_train = train[['hd', 'vo', 'cv']]



sgd_clf = MultiOutputClassifier(SGDClassifier())

_ = sgd_clf.fit(x_train_vec, y_train)

In [7]:
from sklearn import metrics

# **Preprocessing**

In [8]:
#removing URLS an special char
import re

def preprocess_text(text):

    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()
    return text

#preprocessing
train['text'] = train['text'].apply(preprocess_text)
test['text'] = test['text'].apply(preprocess_text)



In [9]:
pip install nltk spacy gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 41.4 MB/s eta 0:00:00


In [10]:
import pandas as pd
import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from gensim.models import Word2Vec


train = pd.read_csv('ghc_train.tsv', sep='\t')
test = pd.read_csv('ghc_test.tsv', sep='\t')

nltk.download('punkt_tab')
nltk.download('stopwords')

# Preprocessing Pipeline
def preprocess_text(text):
    #  Tokenization
    tokens = word_tokenize(text)

    # normalization: Lowercasing and removing punctuation
    normalized = [re.sub(r'[^\w\s]', '', token.lower()) for token in tokens]

    #  removing stopwords
    filtered = [word for word in normalized if word not in stopwords.words('english') and word != '']

    # 4. Stemming
    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(word) for word in filtered]

    return stemmed

# Preprocessing
train['processed_text'] = train['text'].apply(preprocess_text)
test['processed_text'] = test['text'].apply(preprocess_text)

# Word2Vec
all_sentences = train['processed_text'].tolist() + test['processed_text'].tolist()
embedding_model = Word2Vec(sentences=all_sentences, vector_size=100, window=5, min_count=1, workers=4)

train['embeddings'] = train['processed_text'].apply(lambda x: [embedding_model.wv[word] for word in x if word in embedding_model.wv])
test['embeddings'] = test['processed_text'].apply(lambda x: [embedding_model.wv[word] for word in x if word in embedding_model.wv])

# Example output
print(train[['text', 'processed_text', 'embeddings']].head())


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


                                                text  \
0  He most likely converted to islam due to his n...   
1  So Ford lied about being a psychologist. Recor...   
2     Jobs. Education. Ending abuse of Nation. CA43.   
3  I share a lot of your values, & like many who ...   
4  I am so ready to get back to blogging! www.ben...   

                                      processed_text  \
0  [like, convert, islam, due, natur, suitabl, is...   
1  [ford, lie, psychologist, record, seem, indic,...   
2               [job, educ, end, abus, nation, ca43]   
3  [share, lot, valu, like, mani, nt, call, alt, ...   
4  [readi, get, back, blog, wwwbenbrihousecom, re...   

                                          embeddings  
0  [[-0.9412308, 0.9081246, 0.44993803, 0.487285,...  
1  [[-0.15744095, 0.38309354, 0.1842935, 0.179894...  
2  [[-0.28244558, 0.7150143, 0.33958423, 0.319057...  
3  [[-0.31975195, 0.6278179, 0.3024738, 0.2448671...  
4  [[-0.19340108, 0.3797953, 0.18829837, 0.178715..

In [11]:
#vec
# Vectorize the text data
x_train_vec = vectorizer.fit_transform(train['text'])
x_test_vec = vectorizer.transform(test['text'])


# **Training the data**

In [12]:
y_train = train[['hd', 'vo', 'cv']]
y_test = test[['hd', 'vo', 'cv']]


sgd_clf = MultiOutputClassifier(SGDClassifier())
sgd_clf.fit(x_train_vec, y_train)

MultiOutputClassifier(estimator=SGDClassifier())

In [13]:
from sklearn.metrics import accuracy_score

def evaluate(model):
    y_pred = model.predict(x_test_vec)
    for i, col in enumerate(['hd', 'vo', 'cv']):
        accuracy = accuracy_score(test[col], y_pred[:, i])
        print(f'Accuracy for {col}: {100 * accuracy:.2f}%')

evaluate(sgd_clf)

Accuracy for hd: 91.25%
Accuracy for vo: 93.87%
Accuracy for cv: 99.56%


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sklearn.metrics import accuracy_score

train = pd.read_csv('ghc_train.tsv', sep='\t')
test = pd.read_csv('ghc_test.tsv', sep='\t')

x_test = test[['text']]
y_test = test[['cv','hd','vo']]

x_train = train[['text']]
y_train = train[['cv','hd','vo']]

#x_train = test[['text']]
#y_train = train[['hd']]

vectorizer = TfidfVectorizer()
x_train_vec = vectorizer.fit_transform(x_train['text'])
x_test_vec = vectorizer.transform(x_test['text'])

rf = RandomForestClassifier()
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(rf, param_grid, cv=5)
grid_search.fit(x_train_vec, y_train)

best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(x_test_vec)


accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {100*accuracy:.1f}%')


best_rf = grid_search.best_estimator_

Accuracy: 87.3%


Testing on another data set